# Generate Pose2D H36M with YOLO + Key Frame Extraction

Notebook này:
1. Đọc video từ thư mục test_set
2. Chạy YOLO pose → convert COCO 17 keypoints → H36M 17 joints
3. Lưu file `*_h36m_yolo26x.npy` — shape `(frames, 17, 3)` (x, y, conf)
4. Chạy K-Means (n_clusters=30) trên tọa độ 2D → lấy frame gần centroid nhất → `*_key_frames.npy`

Kết quả `.npy` có thể dùng làm dữ liệu đầu vào cho `cali-ai-coach` (qua adapter H36M → 12-point).
Xem thêm: `core/adapter.js` (mediapipeToH36M), `core/pipeline.js`

## 1. Cài đặt thư viện

In [ ]:
!pip -q install -U ultralytics gdown tqdm scikit-learn

## 2. Cấu hình

- `SOURCE_MODE`: `'drive_path'` | `'upload'` | `'gdown'`
- `N_KEY_FRAMES`: số key frame tối đa mỗi video (mặc định 30)
- `RUN_SUBJECTS`: set tên thư mục subject cần xử lý (để trống = tất cả)

In [ ]:
from pathlib import Path

SOURCE_MODE = 'drive_path'
DRIVE_TEST_ROOT = Path('/content/drive/MyDrive/test_set')
DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1FbFj7lz5MkDSKDcjeYoBq6nZoqy29vLS'
RUN_SUBJECTS = {'S3'}
MAX_VIDEOS_THIS_RUN = 756
OVERWRITE_OUTPUTS = False
N_KEY_FRAMES = 30
OUTPUT_ROOT = Path('/content/drive/MyDrive/pose2d_output')

DATA_ROOT = Path('/content/input_videos')
UPLOAD_ROOT = Path('/content/uploaded_videos')

print('Cấu hình OK')
print(f'  SOURCE_MODE      = {SOURCE_MODE}')
print(f'  RUN_SUBJECTS     = {RUN_SUBJECTS}')
print(f'  N_KEY_FRAMES     = {N_KEY_FRAMES}')
print(f'  OUTPUT_ROOT      = {OUTPUT_ROOT}')

## 3. Định nghĩa hàm COCO → H36M

Chuyển đổi 17 keypoints COCO (YOLO output) sang 17 joints H36M.

In [ ]:
import numpy as np

COCO = {
    'nose': 0, 'left_eye': 1, 'right_eye': 2, 'left_ear': 3, 'right_ear': 4,
    'left_shoulder': 5, 'right_shoulder': 6, 'left_elbow': 7, 'right_elbow': 8,
    'left_wrist': 9, 'right_wrist': 10, 'left_hip': 11, 'right_hip': 12,
    'left_knee': 13, 'right_knee': 14, 'left_ankle': 15, 'right_ankle': 16,
}

H36M_NAMES = [
    'Root', 'RHip', 'RKnee', 'RFoot', 'LHip', 'LKnee', 'LFoot', 'Spine',
    'Thorax', 'NeckBase', 'Head', 'LShoulder', 'LElbow', 'LWrist',
    'RShoulder', 'RElbow', 'RWrist',
]


def weighted_point(kps: np.ndarray, names) -> np.ndarray:
    indices = [COCO[x] if isinstance(x, str) else int(x) for x in names]
    pts = kps[indices].astype(np.float32)
    conf = pts[:, 2]
    valid = np.isfinite(pts).all(axis=1) & (conf > 0)
    if not np.any(valid):
        return np.zeros(3, dtype=np.float32)
    pts, conf = pts[valid], conf[valid]
    w = conf / max(float(conf.sum()), 1e-6)
    xy = (pts[:, :2] * w[:, None]).sum(axis=0)
    return np.array([xy[0], xy[1], float(conf.mean())], dtype=np.float32)


def weighted_existing(points) -> np.ndarray:
    pts = np.asarray(points, dtype=np.float32)
    conf = pts[:, 2]
    valid = np.isfinite(pts).all(axis=1) & (conf > 0)
    if not np.any(valid):
        return np.zeros(3, dtype=np.float32)
    pts, conf = pts[valid], conf[valid]
    w = conf / max(float(conf.sum()), 1e-6)
    xy = (pts[:, :2] * w[:, None]).sum(axis=0)
    return np.array([xy[0], xy[1], float(conf.mean())], dtype=np.float32)


def coco_to_h36m(coco_kps: np.ndarray) -> np.ndarray:
    """Convert COCO 17 keypoints → H36M 17 joints (x, y, conf)."""
    if coco_kps.shape[-1] == 2:
        coco_kps = np.concatenate(
            [coco_kps.astype(np.float32), np.ones((coco_kps.shape[0], 1), np.float32)], axis=1)
    coco_kps = coco_kps.astype(np.float32)
    h36m = np.zeros((17, 3), np.float32)
    pelvis = weighted_point(coco_kps, ['left_hip', 'right_hip'])
    shoulder_c = weighted_point(coco_kps, ['left_shoulder', 'right_shoulder'])
    head = weighted_point(coco_kps, ['nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear'])
    h36m[0]  = pelvis
    h36m[1]  = coco_kps[COCO['right_hip']]
    h36m[2]  = coco_kps[COCO['right_knee']]
    h36m[3]  = coco_kps[COCO['right_ankle']]
    h36m[4]  = coco_kps[COCO['left_hip']]
    h36m[5]  = coco_kps[COCO['left_knee']]
    h36m[6]  = coco_kps[COCO['left_ankle']]
    h36m[7]  = weighted_existing([pelvis, shoulder_c])
    h36m[8]  = shoulder_c
    h36m[9]  = weighted_existing([shoulder_c, head])
    h36m[10] = head
    h36m[11] = coco_kps[COCO['left_shoulder']]
    h36m[12] = coco_kps[COCO['left_elbow']]
    h36m[13] = coco_kps[COCO['left_wrist']]
    h36m[14] = coco_kps[COCO['right_shoulder']]
    h36m[15] = coco_kps[COCO['right_elbow']]
    h36m[16] = coco_kps[COCO['right_wrist']]
    return h36m.astype(np.float32)


print('H36M joints:', list(enumerate(H36M_NAMES)))

## 4. Hàm trích xuất Key Frames bằng K-Means

In [ ]:
from sklearn.cluster import KMeans


def extract_key_frames(pose_array: np.ndarray, n_clusters: int = 30) -> np.ndarray:
    """
    Tìm key frames bằng K-Means clustering trên tọa độ 2D H36M.

    1. Trải phẳng (x, y) 17 joint → vector 34 chiều
    2. Chuẩn hóa: trừ pelvis (joint 0) để bất biến vị trí
    3. K-Means → lấy frame gần centroid mỗi cụm
    4. Sắp xếp theo thứ tự thời gian
    """
    F = pose_array.shape[0]
    if F == 0:
        return np.array([], dtype=np.int64)

    mean_conf = pose_array[:, :, 2].mean(axis=1)
    valid_mask = mean_conf > 0
    valid_indices = np.where(valid_mask)[0]

    if len(valid_indices) == 0:
        step = max(1, F // n_clusters)
        return np.arange(0, F, step)[:n_clusters].astype(np.int64)

    xy = pose_array[valid_indices, :, :2].astype(np.float32)
    pelvis = xy[:, 0:1, :]
    xy_norm = xy - pelvis
    vectors = xy_norm.reshape(len(valid_indices), -1)
    vectors = np.nan_to_num(vectors, nan=0.0, posinf=0.0, neginf=0.0)

    k = min(n_clusters, len(valid_indices))
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(vectors)

    centroids = kmeans.cluster_centers_
    labels = kmeans.labels_

    rep_valid_indices = []
    for cluster_id in range(k):
        mask = labels == cluster_id
        if not mask.any():
            continue
        cluster_vectors = vectors[mask]
        cluster_orig_idx = np.where(mask)[0]
        dists = np.linalg.norm(cluster_vectors - centroids[cluster_id], axis=1)
        best = cluster_orig_idx[np.argmin(dists)]
        rep_valid_indices.append(int(valid_indices[best]))

    return np.array(sorted(set(rep_valid_indices)), dtype=np.int64)

## 5. Chạy YOLO pose và lưu kết quả

Quét qua từng video, chạy YOLO pose estimation, chuyển sang H36M, lưu `.npy`.

In [ ]:
import os, shutil, zipfile, csv, cv2, torch
from tqdm.auto import tqdm
from ultralytics import YOLO

MODEL_NAME = 'yolo26x-pose.pt'
IMG_SIZE = 640
CONF_THRES = 0.25
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO(MODEL_NAME)

VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv', '.m4v', '.wmv'}

In [ ]:
def find_root_with_subjects(root: Path) -> Path:
    root = Path(root)
    subject_dirs = [p for p in root.iterdir() if p.is_dir() and p.name.upper().startswith('S')]
    if subject_dirs:
        return root
    if any(p.suffix.lower() in VIDEO_EXTS for p in root.iterdir() if p.is_file()):
        return root
    candidates = sorted(
        [p for p in root.rglob('*') if p.is_dir() and p.name.upper().startswith('S')],
        key=lambda p: len(p.parts)
    )
    if candidates:
        return candidates[0].parent
    return root


def prepare_input() -> Path:
    if SOURCE_MODE == 'gdown':
        import gdown
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        print(f'Downloading from Drive to {DATA_ROOT}...')
        gdown.download_folder(url=DRIVE_FOLDER_URL, output=str(DATA_ROOT),
                              quiet=False, use_cookies=False)
        return find_root_with_subjects(DATA_ROOT)
    elif SOURCE_MODE == 'drive_path':
        from google.colab import drive
        drive.mount('/content/drive')
        test_root = Path(DRIVE_TEST_ROOT)
        if not test_root.exists():
            raise FileNotFoundError(f'DRIVE_TEST_ROOT not found: {test_root}')
        return test_root
    elif SOURCE_MODE == 'upload':
        from google.colab import files
        UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
        uploaded = files.upload()
        zip_files = [n for n in uploaded if n.lower().endswith('.zip')]
        if zip_files:
            zip_path = Path('/content') / zip_files[0]
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(UPLOAD_ROOT)
        else:
            for name in uploaded:
                src = Path('/content') / name
                dst = UPLOAD_ROOT / name
                if src.exists():
                    shutil.move(str(src), str(dst))
        return UPLOAD_ROOT
    else:
        raise ValueError("SOURCE_MODE must be 'gdown', 'drive_path', or 'upload'")


TEST_ROOT = prepare_input()
print(f'TEST_ROOT = {TEST_ROOT}')

In [ ]:
def count_video_frames(path: Path):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    return total if total > 0 else None


def pick_best_person(result):
    if result.keypoints is None or result.keypoints.data is None:
        return None
    data = result.keypoints.data.detach().cpu().numpy().astype(np.float32)
    if data.size == 0:
        return None
    if data.shape[-1] == 2:
        conf = np.ones(data.shape[:2] + (1,), np.float32)
        data = np.concatenate([data, conf], axis=-1)
    scores = np.nanmean(data[:, :, 2], axis=1)
    if not np.isfinite(scores).any():
        return None
    return data[int(np.nanargmax(scores))]


def pose_output_path(video_path: Path) -> Path:
    try:
        rel = video_path.relative_to(TEST_ROOT)
    except ValueError:
        rel = Path(video_path.name)
    return OUTPUT_ROOT / rel.parent / f'{video_path.stem}_h36m_yolo26x.npy'


def keyframe_output_path(video_path: Path) -> Path:
    try:
        rel = video_path.relative_to(TEST_ROOT)
    except ValueError:
        rel = Path(video_path.name)
    return OUTPUT_ROOT / rel.parent / f'{video_path.stem}_key_frames.npy'


def process_video(video_path: Path) -> np.ndarray:
    total = count_video_frames(video_path)
    frames = []
    stream = model.predict(
        source=str(video_path), stream=True,
        imgsz=IMG_SIZE, conf=CONF_THRES, device=DEVICE, verbose=False
    )
    for result in tqdm(stream, total=total, desc=video_path.name, leave=False):
        person = pick_best_person(result)
        frames.append(coco_to_h36m(person) if person is not None
                      else np.zeros((17, 3), np.float32))
    if not frames:
        return np.empty((0, 17, 3), np.float32)
    return np.stack(frames).astype(np.float32)


# Lọc danh sách video
all_videos = sorted(
    [p for p in TEST_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXTS],
    key=lambda p: str(p).lower()
)

if RUN_SUBJECTS:
    def belongs(p: Path) -> bool:
        try:
            parts = p.relative_to(TEST_ROOT).parts
        except ValueError:
            parts = (p.parent.name,)
        return bool(parts) and parts[0] in RUN_SUBJECTS
    videos = [p for p in all_videos if belongs(p)]
else:
    videos = all_videos

print(f'Total videos: {len(videos)}')
print(f'Device: {DEVICE}')

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for video_path in videos[:MAX_VIDEOS_THIS_RUN]:
    pose_path = pose_output_path(video_path)
    kf_path = keyframe_output_path(video_path)
    pose_path.parent.mkdir(parents=True, exist_ok=True)

    # Pose extraction
    if OVERWRITE_OUTPUTS or not pose_path.exists():
        arr = process_video(video_path)
        np.save(pose_path, arr)
        print(f'[pose] {pose_path.name}  shape={arr.shape}')
    else:
        arr = np.load(pose_path)

    # Key frame extraction
    if OVERWRITE_OUTPUTS or not kf_path.exists():
        if arr.shape[0] > 0:
            key_frames = extract_key_frames(arr, n_clusters=N_KEY_FRAMES)
        else:
            key_frames = np.array([], dtype=np.int64)
        np.save(kf_path, key_frames)
        print(f'[kf] {kf_path.name}  n={len(key_frames)}')

print('Done!')

## 6. Xuất JSON 17 điểm từ file NPY

Sau khi có file `.npy` từ YOLO, dùng các hàm dưới đây để xuất file JSON 17 điểm sạch (đã trừ pelvis) cho Node.js pipeline.

In [ ]:
import json
import numpy as np


def export_pure_h36m_json(npy_pose_path, output_json_path, exercise_name="push_up", is_standard=True):
    """Xuất toàn bộ chuỗi frame của video thành mảng JSON 17 điểm sạch (đã trừ pelvis)."""
    pose_data = np.load(npy_pose_path)  # (frames, 17, 3)

    json_frames = []
    for idx, frame_joints in enumerate(pose_data):
        px, py = frame_joints[0, 0], frame_joints[0, 1]
        landmarks_17 = []
        for joint in frame_joints:
            landmarks_17.append({
                "x": float(joint[0]) - px,
                "y": float(joint[1]) - py,
                "z": 0.0,
            })
        json_frames.append({
            "frame": idx,
            "landmarks": landmarks_17
        })

    output_data = {
        "exercise": exercise_name,
        "type": "standard" if is_standard else "student",
        "frames": json_frames
    }
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    print(f"✅ Đã xuất JSON: {output_json_path}  ({len(json_frames)} frames)")


def export_keyframes_only(npy_pose_path, npy_kf_path, output_json_path):
    """Lọc riêng tọa độ của 30 trạm mẫu từ K-Means để làm Checkpoints (đã trừ pelvis)."""
    pose_data = np.load(npy_pose_path)  # (frames, 17, 3)
    key_frames = np.load(npy_kf_path).astype(int).tolist()

    kf_poses = []
    for kf_idx in key_frames:
        frame_joints = pose_data[kf_idx]
        px, py = frame_joints[0, 0], frame_joints[0, 1]
        landmarks_17 = []
        for joint in frame_joints:
            landmarks_17.append({
                "x": float(joint[0]) - px,
                "y": float(joint[1]) - py,
                "z": 0.0,
            })
        kf_poses.append(landmarks_17)

    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(kf_poses, f, ensure_ascii=False, indent=2)
    print(f"🎯 Đã xuất Keyframes Checkpoints: {output_json_path}  ({len(kf_poses)} frames)")


# --- Ví dụ thực thi ---
# Chỉnh đường dẫn file .npy cho phù hợp với output của notebook
# export_keyframes_only('video_chuan_h36m.npy', 'video_chuan_key_frames.npy', 'push_up_keyframes.json')
# export_pure_h36m_json('video_nguoitap_h36m.npy', 'student_full_stream.json', is_standard=False)
print("Hàm xuất JSON đã sẵn sàng. Bỏ comment 2 dòng cuối để chạy.")